In [1]:
# part 1
from flask import Flask,request,render_template
import pickle
import numpy as np
import pandas as pd

In [2]:
#part 2
# create an instance of our web application
app = Flask(__name__)

In [3]:

#part 3
#load model 
OHE_path = r'E:\Info_Srping\FWI_HMI\Deploy_HMI\Model\one_hot_encoder.pkl'
PT_path = r'E:\Info_Srping\FWI_HMI\Deploy_HMI\Model\power_transformers.pkl'
RR_path = r'E:\Info_Srping\FWI_HMI\Deploy_HMI\Model\Ridge_regression_V2.pkl'
SS_path = r'E:\Info_Srping\FWI_HMI\Deploy_HMI\Model\standard_scaler.pkl'

model = None #initialize as None
std_scale = None
power_trans = None
ohe = None

try:
    with open(OHE_path, 'rb') as f:
        ohe = pickle.load(f)
    print('Load successfully')
    with open(PT_path, 'rb') as f:
        power_trans = pickle.load(f)
    print('Load successfully')
    with open(SS_path, 'rb') as f:
        std_scale = pickle.load(f)
    print('Load successfully')
    with open(RR_path, 'rb') as f:
        model = pickle.load(f)
    print('Load successfully')
except FileNotFoundError:
    print('file not found')
except Exception as e:
    print(f'error {e}')


Load successfully
Load successfully
Load successfully
Load successfully


In [5]:
def inverse_single_column(scaler, values, col_idx):
    """Undo StandardScaler for one column, keep others zeroed."""
    n_features = len(scaler.mean_)
    dummy = np.zeros((len(values), n_features))
    dummy[:, col_idx] = values
    return scaler.inverse_transform(dummy)[:, col_idx]


In [4]:
@app.route('/', methods=['GET', 'POST'])
def index():
    prediction_result = None

    if request.method == 'POST' and model:
        try:
            print("---- Step 1: Collecting inputs ----")
            raw_values = {
                'temperature':             float(request.form['feature_input_1']),
                'dew_point_temperature':   float(request.form['feature_input_2']),
                'relative_humidity':       float(request.form['feature_input_3']),
                'atmospheric_pressure':    float(request.form['feature_input_4']),
                'precipitation':           float(request.form['feature_input_5']),
                'rain_days':               int(request.form['feature_input_6']),
                'country':                 request.form['feature_input_7'],
                'fine_fuel_moisture_code': float(request.form['feature_input_8']),
                'duff_moisture_code':      float(request.form['feature_input_9']),
                'drought_code':            float(request.form['feature_input_10']),
                'build_up_index':          float(request.form['feature_input_11']),
                'initial_spread_index':    float(request.form['feature_input_12']),
                'fire_weather_index':      0,  # dummy target
                'wind_speed':              float(request.form['feature_input_13'])
            }
            print("Raw inputs:", raw_values)

            print("---- Step 2: Create DataFrame ----")
            new_data = pd.DataFrame([raw_values])
            print(new_data.head())

            print("---- Step 3: Apply PowerTransforms ----")
            for col, trans in power_trans.items():
                new_data[col + '_transformed'] = trans.transform(new_data[[col]].values)
            print(new_data.head())

            print("---- Step 4: Apply StandardScaler ----")

            new_data = new_data[['temperature', 'dew_point_temperature', 'relative_humidity',
               'wind_speed', 'atmospheric_pressure', 'precipitation', 'rain_days',
               'fine_fuel_moisture_code', 'duff_moisture_code', 'drought_code',
               'build_up_index', 'initial_spread_index', 'fire_weather_index',
               'fine_fuel_moisture_code_transformed', 'duff_moisture_code_transformed',
               'drought_code_transformed', 'build_up_index_transformed',
               'initial_spread_index_transformed', 'fire_weather_index_transformed',
               'wind_speed_transformed']]
                    
            # 2. Apply scaling
            scaled_array = std_scale.transform(new_data)
            
            # 3. Convert back to DataFrame with same column names
            new_data = pd.DataFrame(scaled_array, columns=new_data.columns)
            
            print("After scaling:", new_data.head())
            

            print("---- Step 5: Reorder + Feature Engineering ----")
            new_data = new_data[['temperature', 'dew_point_temperature', 'relative_humidity',
                                 'atmospheric_pressure', 'precipitation', 'rain_days',
                                 'fine_fuel_moisture_code_transformed',
                                 'duff_moisture_code_transformed', 'drought_code_transformed',
                                 'build_up_index_transformed', 'initial_spread_index_transformed',
                                 'fire_weather_index_transformed', 'wind_speed_transformed']]
            print(new_data.head())

            # 5 Feature engineering ──────────────────────────────────────── 
            new_data['FFMC__TEMP'] = new_data['temperature'] * new_data['fine_fuel_moisture_code_transformed'] 
            new_data['FFMC_RH'] = new_data['relative_humidity'] * new_data['fine_fuel_moisture_code_transformed'] 
            new_data['RH_TEMP'] = new_data['temperature'] * new_data['relative_humidity'] 
            new_data['RD_FFMC'] = new_data['fine_fuel_moisture_code_transformed'] * new_data['rain_days'] 
            new_data['DC_RD'] = new_data['drought_code_transformed'] * new_data['rain_days'] 
            new_data['WS_TENP'] = new_data['wind_speed_transformed'] * new_data['fine_fuel_moisture_code_transformed'] 
            new_data['IST_WS'] = new_data['wind_speed_transformed'] * new_data['initial_spread_index_transformed'] 
            new_data['ISI_FFMC'] = new_data['initial_spread_index_transformed'] * new_data['fine_fuel_moisture_code_transformed'] 
            new_data['ISI_RH'] =  new_data['initial_spread_index_transformed'] * new_data['relative_humidity']
            new_data['DMC_TEMP'] = new_data['duff_moisture_code_transformed'] * new_data['temperature'] 
            new_data['DMC_RD'] = new_data['duff_moisture_code_transformed'] * new_data['rain_days'] 
            new_data['RD_TEMP'] = new_data['rain_days'] * new_data['temperature'] 
            new_data['DMC_DC'] = new_data['drought_code_transformed'] * new_data['duff_moisture_code_transformed'] 
            new_data['BUI_DC'] = new_data['drought_code_transformed'] * new_data['build_up_index_transformed'] 
            new_data['BUI_DMC'] = new_data['duff_moisture_code_transformed'] * new_data['build_up_index_transformed']
            print(new_data.head())
          
            # Step 1: Keep country in new_data
            
            new_data['country'] = request.form['feature_input_7']
            print(new_data.head())
            
            # Step 2: One-hot encode
            print("starting of ohe")
            encoded = pd.DataFrame(ohe.transform(new_data[['country']]),
                columns=ohe.get_feature_names_out(['country'])
            )

            # 3. Concatenate with new_data (keep the raw 'country' if model needs it)
            new_data = pd.concat([new_data.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
            new_data = new_data.drop(['country'], axis =1)
            print(new_data.head())

            new_data['BUI_SQRT'] = (new_data['build_up_index_transformed'].clip(lower=0)) ** 0.5
            new_data['FFMC_SQRT'] = (new_data['fine_fuel_moisture_code_transformed'].clip(lower=0)) ** 0.5
            new_data['ISI_SQRT'] = (new_data['initial_spread_index_transformed'].clip(lower=0)) ** 0.5


            print("---- Step 7: Final Prediction ----")
           
            X = new_data.drop('fire_weather_index_transformed', axis=1, errors='ignore')
            y_scaled = model.predict(X)[0]

          
            """
            # ---- Step 7: Bring prediction back to original scale ----
            col_index = list(std_scale.feature_names_in_).index('fire_weather_index_transformed')
            y_power = inverse_single_column(std_scale, np.array([y_scaled]), col_index, n_features)
            
            # pick the right transformer from dict
            col_index = list(power_trans.feature_names_in_).index('fire_weather_index')
            pt_target = power_trans['fire_weather_index_transformed']  # or 'fire_weather_index'
            y_original = pt_target.inverse_transform(y_power.reshape(-1, 1))[0, 0]
            
            prediction_result = f"Predicted FWI: {y_original:.2f}"


            pt_target = power_trans['fire_weather_index']
            y_original = pt_target.inverse_transform(y_power.reshape(-1, 1))[0, 0]
          
            prediction_result = f"Predicted FWI: {y_original:.2f}"
            print("Prediction:", prediction_result)
            """
            
            # ---- Step 7: Bring prediction back to original scale ----
            col_index = list(std_scale.feature_names_in_).index('fire_weather_index_transformed')
            y_power = inverse_single_column(std_scale, np.array([y_scaled]), col_index)
            
            # Pick transformer from dict
            pt_target = power_trans['fire_weather_index']
            y_original = pt_target.inverse_transform(y_power.reshape(-1, 1))[0, 0]
            
            prediction_result = f"Predicted FWI: {y_original:.2f}"

        
        except Exception as e:
            print("Exception occurred:", e)
            prediction_result = f"Error: {e}"

    return render_template('index.html', prediction_result=prediction_result)


In [6]:
#part 6 Run the Flask application
#this block ensures that flask app starts when you run the python script directly
#debug = True is great for development as it automatically reloads the server on code changes
#and provide detailed error message. Disable this in production!

#if __name__ =='main':
#    app.run(debug = True)
# Cell 3: Run the Flask App
# This will block the cell's execution until the server is stopped.
# You can specify a port if 5000 is already in use: app.run(debug=True, port=5001)

app.run(debug=True, host = "0.0.0.0", port = 5000,use_reloader=False, )

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.96.204:5000
Press CTRL+C to quit
127.0.0.1 - - [29/Sep/2025 21:15:09] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [29/Sep/2025 21:15:10] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [29/Sep/2025 21:15:10] "GET /favicon.ico HTTP/1.1" 404 -


---- Step 1: Collecting inputs ----
Raw inputs: {'temperature': 2.0, 'dew_point_temperature': 2.0, 'relative_humidity': 2.0, 'atmospheric_pressure': 2.0, 'precipitation': 2.0, 'rain_days': 2, 'country': 'Canada', 'fine_fuel_moisture_code': 2.0, 'duff_moisture_code': 2.0, 'drought_code': 2.0, 'build_up_index': 2.0, 'initial_spread_index': 2.0, 'fire_weather_index': 0, 'wind_speed': 2.0}
---- Step 2: Create DataFrame ----
   temperature  dew_point_temperature  relative_humidity  \
0          2.0                    2.0                2.0   

   atmospheric_pressure  precipitation  rain_days country  \
0                   2.0            2.0          2  Canada   

   fine_fuel_moisture_code  duff_moisture_code  drought_code  build_up_index  \
0                      2.0                 2.0           2.0             2.0   

   initial_spread_index  fire_weather_index  wind_speed  
0                   2.0                   0         2.0  
---- Step 3: Apply PowerTransforms ----
   temperature 

127.0.0.1 - - [29/Sep/2025 21:15:43] "POST / HTTP/1.1" 200 -


   temperature  dew_point_temperature  relative_humidity  \
0    -1.863211              -0.715795          -2.647197   

   atmospheric_pressure  precipitation  rain_days  \
0           -136.682106      -0.070755  -0.109146   

   fine_fuel_moisture_code_transformed  duff_moisture_code_transformed  \
0                            -2.022928                       -1.331931   

   drought_code_transformed  build_up_index_transformed  ...   WS_TENP  \
0                 -1.834988                    -1.52413  ...  3.742711   

     IST_WS  ISI_FFMC    ISI_RH  DMC_TEMP    DMC_RD   RD_TEMP    DMC_DC  \
0  0.506488  0.553788  0.724685  2.481669  0.145375  0.203362  2.444078   

     BUI_DC   BUI_DMC  
0  2.796761  2.030036  

[1 rows x 28 columns]
   temperature  dew_point_temperature  relative_humidity  \
0    -1.863211              -0.715795          -2.647197   

   atmospheric_pressure  precipitation  rain_days  \
0           -136.682106      -0.070755  -0.109146   

   fine_fuel_moisture_co

127.0.0.1 - - [29/Sep/2025 21:15:43] "GET /static/style.css HTTP/1.1" 304 -


In [202]:
print("Available keys in power_trans:", power_trans.keys())

Available keys in power_trans: dict_keys(['fine_fuel_moisture_code', 'duff_moisture_code', 'drought_code', 'build_up_index', 'initial_spread_index', 'fire_weather_index', 'wind_speed'])
